In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [6]:
# df_green.registerTempTable('green')
# 이 방식은 너무 오래됐으니 새 방식을 써라
df_green.createOrReplaceTempView('green')

In [14]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [15]:
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-17 20:00:00|  41| 633.6499999999997|            66|
|2020-01-03 18:00:00| 223| 313.6600000000001|            25|
|2020-01-22 07:00:00|  33|            185.72|            10|
|2020-01-26 20:00:00|  89|46.879999999999995|             2|
|2020-01-05 10:00:00| 244|241.14000000000001|            15|
|2020-01-08 05:00:00| 167|143.48000000000002|             5|
|2020-01-03 10:00:00| 106|             32.39|             2|
|2020-01-16 11:00:00|   7|264.74000000000007|            20|
|2020-01-21 12:00:00| 205|205.97000000000003|             5|
|2020-01-12 17:00:00| 177|             86.08|             2|
|2020-01-03 22:00:00|  74| 657.7199999999998|            46|
|2020-01-09 15:00:00| 174|493.66999999999996|            15|
|2020-01-24 20:00:00| 210|               7.3|             1|
|2020-01-27 18:00:00|  1

In [16]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

In [17]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.registerTempTable('yellow')

In [18]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [24]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

In [25]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [26]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [27]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [28]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [29]:
df_join = spark.read.parquet('data/report/revenue/total')

In [30]:
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

In [32]:
df_zones = spark.read.parquet('zones/')

In [33]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [34]:
df_result.drop('LocationID', 'zone').write.mode('overwrite').parquet('tmp/revenue-zones')